# Data analysis

A lightweight, schema-tolerant overview of a `reddit-minerals` JSONL or JSON export. Treat all model-derived labels and scores as estimates, not ground truth.

In [ ]:
MINERAL = "gold"
EXPORT_PATH = "exports/gold.jsonl"

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

working_directory = Path.cwd().resolve()
project_root = (
    working_directory.parent if working_directory.name == "notebooks" else working_directory
)
export_path = Path(EXPORT_PATH)
if not export_path.is_absolute():
    export_path = project_root / export_path

In [ ]:
records = []
if export_path.exists() and export_path.suffix.casefold() == ".jsonl":
    with export_path.open(encoding="utf-8") as export_file:
        records = [json.loads(line) for line in export_file if line.strip()]
elif export_path.exists():
    with export_path.open(encoding="utf-8") as export_file:
        document = json.load(export_file)
    records = document.get("records", []) if isinstance(document, dict) else document
else:
    print(f"No export found at {export_path}")

frame = pd.json_normalize(records, sep="_") if records else pd.DataFrame()
if "mineral" in frame.columns:
    frame = frame.loc[frame["mineral"].astype("string").str.casefold() == MINERAL.casefold()].copy()
print(f"Analyzing {len(frame):,} records across {len(frame.columns):,} fields")

In [ ]:
dimension_columns = [
    column
    for column in ("record_type", "kind", "subreddit", "status", "mineral")
    if column in frame.columns
]
if frame.empty:
    display(pd.DataFrame({"message": ["Create an export to populate this report."]}))
elif dimension_columns:
    for column in dimension_columns:
        display(frame[column].value_counts(dropna=False).rename("count").to_frame())
else:
    display(frame.head())

In [ ]:
numeric = frame.select_dtypes(include="number")
if numeric.empty:
    print("No numeric fields are available for summary.")
else:
    display(numeric.describe().T)
    plot_columns = numeric.columns[:6]
    numeric.loc[:, plot_columns].hist(figsize=(12, 7), bins=20)
    plt.suptitle(f"Numeric distributions: {MINERAL}")
    plt.tight_layout()

## Interpretation limits

Sampling choices, subreddit selection, deleted content, API availability, and model behavior all affect these distributions. Do not infer population-level public opinion or individual credibility from this notebook. See `docs/methodology.md` and `docs/privacy-compliance.md`.